# Quick, Draw!

Результаты завершённых экспериментов на полном validation. Обучение выполняется через `train.py` и `improve.py`; `verify.py` заново проверяет сохранённые веса и submission.

In [1]:
import json
from pathlib import Path
import pandas as pd

folders = [Path("output") / name for name in [
    "run-20261014", "resize56-20261014", "regularized-20261014", "regularized-fixed-20261014", "ensemble-20261014"
]]
rows = []
for folder in folders:
    if (folder / "summary.json").exists():
        summary = json.loads((folder / "summary.json").read_text())
        rows.append({"run": folder.name, "model": summary["selected"]["model"],
                     "accuracy": summary["selected"]["accuracy"]})
results = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
results

,run,model,accuracy
3,ensemble-20261014,"mean_(0, 1, 2, 3)_t4_dNone",0.749111
1,resize56-20261014,ensemble,0.743894
2,regularized-fixed-20261014,parent,0.742164
0,run-20261014,ensemble_tta,0.742126


In [2]:
baseline = pd.read_csv("output/run-20261014/scores.csv")
baseline[baseline["model"] == "baseline"]

,model,accuracy
0,baseline,0.678087


In [3]:
best_output = Path("output") / results.iloc[0]["run"]
pd.read_csv(best_output / "scores.csv").sort_values("accuracy", ascending=False).head(10)

,model,correct,accuracy
227,"mean_(0, 1, 2, 3)_t4_dNone",77533,0.749111
216,"mean_(1, 2, 3)_t4_d3",77501,0.748802
213,"mean_(1, 2, 3)_t4_dNone",77487,0.748667
222,"mean_(0, 1, 2, 3)_t2_dNone",77486,0.748657
212,"mean_(1, 2, 3)_t2_d3",77468,0.748483
209,"mean_(1, 2, 3)_t2_dNone",77461,0.748415
217,"mean_(0, 1, 2, 3)_t1_dNone",77438,0.748193
208,"mean_(1, 2, 3)_t1_d3",77424,0.748058
178,"mean_(0, 1, 2)_t4_d0",77422,0.748039
177,"mean_(0, 1, 2)_t4_dNone",77413,0.747952


In [4]:
verification = best_output / "verification.json"
if verification.exists():
    display(json.loads(verification.read_text()))

{'correct': 77533,
 'val_examples': 103500,
 'accuracy': 0.7491111111111111,
 'recorded_accuracy': 0.7491111111111111,
 'submission_rows': 172500,
 'submission_differences': 0}

## Submission

In [5]:
submission = pd.read_csv(best_output / "submission.csv")
print(len(submission))
submission.head()

172500


,id,label
0,0,63
1,1,136
2,2,292
3,3,310
4,4,6


В условии порог на 6 баллов — строго выше 0.76102, на 8 — строго выше 0.77489. Отдельного порога на 7 нет. Validation используется для выбора модели; баллы подтверждаются результатом на Kaggle.